A jupyter notebook to interpolate data to z=300m for visulations.

In [72]:
import numpy as np
from netCDF4 import Dataset
from matplotlib import pyplot as plt
import matplotlib
import matplotlib.colors as colors

In [73]:
def z_interp(h, field_vals_all, lon, lat, z_val):
    field_vals = np.zeros((len(lat), len(lon)))
    for i in np.arange(len(lat)):
        for j in np.arange(len(lon)):
            if h[-1,i,j] > z_val:
                # This value is inside the topography
                field_vals[i,j] = np.nan
            else:
                # Find indices either side of this value
                low_idx = np.where(h[:,i,j] < z_val)[0][0]
                high_idx = np.where(h[:,i,j] > z_val)[0][-1]

                # Compute weightings
                weight_low = (z_val - h[low_idx,i,j])/(h[high_idx,i,j] - h[low_idx,i,j])
                weight_high = 1. - weight_low

                # Compute and store value
                field_vals[i,j] = weight_low*field_vals_all[low_idx, i, j] + weight_high*field_vals_all[high_idx, i, j]
    return field_vals

def cubic_z_interp(h, field_vals_all, lon, lat, z_val):
    # Now make cubic interpolation coefficients for each grid staggering

    # Use the bottom four levels (in Python notation!)
    levels = [-1, -2, -3, -4]
    
    coeffs = np.zeros((4, len(lat), len(lon)))

    # Compute weights using interpolating polynomials
    coeffs[0] = (
        (z_val - h[levels[1]]) * (z_val - h[levels[2]])
        * (z_val - h[levels[3]])
    ) / (
        (h[levels[0]] - h[levels[1]]) * (h[levels[0]] - h[levels[2]])
        * (h[levels[0]] - h[levels[3]])
    )
    coeffs[1] = (
        (z_val - h[levels[0]]) * (z_val - h[levels[2]])
        * (z_val - h[levels[3]])
    ) / (
        (h[levels[1]] - h[levels[0]]) * (h[levels[1]] - h[levels[2]])
        * (h[levels[1]] - h[levels[3]])
    )
    coeffs[2] = (
        (z_val - h[levels[0]]) * (z_val - h[levels[1]])
        * (z_val - h[levels[3]])
    ) / (
        (h[levels[2]] - h[levels[0]]) * (h[levels[2]] - h[levels[1]])
        * (h[levels[2]] - h[levels[3]])
    )
    coeffs[3] = (
        (z_val - h[levels[0]]) * (z_val - h[levels[1]])
        * (z_val - h[levels[2]])
    ) / (
        (h[levels[3]] - h[levels[0]]) * (h[levels[3]] - h[levels[1]])
        * (h[levels[3]] - h[levels[2]])
    )

    field_vals = np.zeros((len(lat), len(lon)))
    
    for i in np.arange(4):
        field_vals += coeffs[i]*field_vals_all[levels[i]]

    # Set values below surface to NaN
    field_vals = np.where(
        h[levels[0]] > z_val, np.nan, field_vals
    )
    
    return field_vals

In [74]:
# Choose the data to regrid
test = 'gap'
#test='vortex'

rot = False

dycore = 'CAM-SE'
#dycore = 'CAM-FV3'
#dycore = 'CAM-MPAS'

# Choose the latitude (index) to take the cross section at
# For C192, lat_val = 200 => 10 deg N
lat_val = 200

In [75]:
# Choose the latitude (index) to take the cross section at
# For C192, lat_val = 200 => 10 deg N
lat_val = 200

# Choose RF damping name

if rot:
    rot_state='with_rot'
else:
    rot_state='omega0'

# The three options I have

# No Rayleigh damping
case = f'cam_6_4_100_se_ne60_ztop20km_L57'
nc_file = f'{case}.cam.h0i.0001-01-01-00000_{test}_{rot_state}_tau_0.nc'
extra_savename = 'tau_0'

# Damping timescale is 100 s
#case = f'cam_6_4_100_se_ne60_ztop20km_L57_new_RF'
#nc_file = f'{case}.cam.h0i.0001-01-01-00000_{test}_{rot_state}_tau_100s.nc'
#extra_savename = 'tau_100s'

# Damping timescale is 40 s 
#case = f'cam_6_4_100_se_ne60_ztop20km_L57_new_RF'
#nc_file = f'{case}.cam.h0i.0001-01-01-00000_{test}_{rot_state}_tau_40s.nc'
#extra_savename = 'tau_40s'


run_base = "/glade/derecho/scratch/timand/"
run_path = run_base + case + '/run/' + nc_file
nc = Dataset(run_path)

In [76]:
# Save the regridded data. Do so as net cdf
savename = f'{dycore}_{test}_{rot_state}_vert_slice_{extra_savename}'
output_file = Dataset(f'/glade/u/home/timand/dcmip2025_gap_and_vortex/interpolate_data/interp_data/{savename}.nc', 'w')

time = nc['time'][:]
lat = nc['lat'][:] 
lon = nc['lon'][:]
lev = nc['lev'][:]

output_file.createDimension('lon', len(lon))
output_file.createDimension('lat', len(lat))
lon_var = output_file.createVariable('lon', 'f4', ('lon',))
lat_var = output_file.createVariable('lat', 'f4', ('lat',))

output_file.variables['lon'][:] = lon
output_file.variables['lat'][:] = lat

output_file.createDimension('time', len(time))
time_var = output_file.createVariable('time', 'f4', ('time',))
output_file.variables['time'][:] = time

output_file.createDimension('lev', len(lev))
lev_var = output_file.createVariable('lev', 'f4', ('lev',))
output_file.variables['lev'][:] = lev

In [77]:
omega_var = output_file.createVariable('OMEGA', 'f4', ('time', 'lev', 'lon'))
T_var = output_file.createVariable('T', 'f4', ('time', 'lev', 'lon'))

for t_idx in np.arange(len(time)):

    print(t_idx)

    OMEGA_field_vals = nc['OMEGA'][t_idx, :, lat_val, :]
    output_file.variables['OMEGA'][t_idx, :, :] = OMEGA_field_vals

    T_field_vals = nc['T'][t_idx, :, lat_val, :]
    output_file.variables['T'][t_idx, :, :] = T_field_vals

output_file.close()

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
